# Query Calibration & Texonomy Audit

**Author:** Samuele Tommasi

## 1. Notebook Objective

This notebook calibrates the patent retrieval strategy for the Unmanned Aerial Vehicle (UAV) sector. All queries are restricted to substantive Patents of Invetion (excluding utility models) filed between 2005 and 2025, aggregated at the DOCDB family level.

In [1]:
from pathlib import Path

import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import numpy as np
import polars as pl
from epo.tipdata.patstat import PatstatClient
from epo.tipdata.patstat.database.models import (
    TLS201_APPLN,
    TLS202_APPLN_TITLE,
    TLS203_APPLN_ABSTR,
    TLS209_APPLN_IPC,
    TLS224_APPLN_CPC,
)
from sentence_transformers import SentenceTransformer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import normalize
from sqlalchemy import and_, distinct, func, or_

ENV = "PROD"
IPR_TYPE = "PI"
YEAR_WINDOW = (2004, 2025)
CACHE_DIR = Path("../data/raw")
CACHE_DIR.mkdir(parents=True, exist_ok=True)

patstat = PatstatClient(env=ENV)
db = patstat.orm()

## 2. Core Taxonomy & IPC Reclassification Audit

The unconditional core UAV set comprises classes restricted exclusively to unmanned vehicles by official definition:
- B64U (unmanned aerial vehicle and equipment therefor)
- B64C 39/024 in CPC (of the remote controlled vehicle type, i.e. RPV) mapped to B64C 39/02 in IPC.
- G08G 5/57 and G08G 5/27 (traffic control systems for aircraft, specifically for UAVs)
- G05D 1/228 (command input arrangements located on-board unmanned vehicles)

To account for reclassification lags across national patent officies, we evaluate family coverage across both CPC and IPC.

In [2]:
CORE_PREFIXES = ("B64U%", "G08G5/57%", "G08G5/27%", "G05D1/228%")

def fetch_families(table, col, scheme: str) -> pl.DataFrame:
    clean_sym = func.replace(col, " ", "")
    b64c_prefix = "B64C39/024%" if scheme == "CPC" else "B64C39/02%"
    prefixes = (*CORE_PREFIXES, b64c_prefix)

    q = (
        db.query(TLS201_APPLN.docdb_family_id, TLS201_APPLN.appln_auth)
            .select_from(table)
            .join(TLS201_APPLN, table.appln_id == TLS201_APPLN.appln_id)
            .filter(
                TLS201_APPLN.ipr_type == IPR_TYPE,
                TLS201_APPLN.earliest_filing_year.between(*YEAR_WINDOW),
                or_(*(clean_sym.like(p) for p in prefixes)),
            )
            .distinct()
    )

    return pl.from_pandas(patstat.df(q))

df_cpc = fetch_families(TLS224_APPLN_CPC, TLS224_APPLN_CPC.cpc_class_symbol, "CPC")
df_ipc = fetch_families(TLS209_APPLN_IPC, TLS209_APPLN_IPC.ipc_class_symbol, "IPC")

coverage_audit = (
    df_cpc.join(df_ipc, on=["docdb_family_id", "appln_auth"], how="full", coalesce=True)
        .with_columns(
            in_cpc=pl.col("docdb_family_id").is_in(df_cpc["docdb_family_id"]),
            in_ipc=pl.col("docdb_family_id").is_in(df_ipc["docdb_family_id"]),
        )
        .group_by("appln_auth")
        .agg(
            pl.col("docdb_family_id").filter(pl.col("in_cpc")).len().alias("cpc_families"),
            pl.col("docdb_family_id").filter(pl.col("in_ipc") & ~pl.col("in_cpc")).len().alias("ipc_only_families"),
            pl.len().alias("union_families")
        )
        .with_columns(
            (pl.col("ipc_only_families") / pl.col("union_families") * 100)
            .round(2)
            .alias("ipc_only_share_pct")
        )
        .sort("union_families", descending=True)
)

core_family_ids = pl.concat([df_cpc.select("docdb_family_id"), df_ipc.select("docdb_family_id")]).unique()
print(f"Distinct core UAV docdb families (CPC U IPC): {len(core_family_ids):,}")

coverage_audit.head(10)

Distinct core UAV docdb families (CPC U IPC): 61,967


appln_auth,cpc_families,ipc_only_families,union_families,ipc_only_share_pct
str,u32,u32,u32,f64
"""CN""",26191,7725,33916,22.78
"""US""",14658,346,15004,2.31
"""WO""",10112,589,10701,5.5
"""KR""",8154,1833,9987,18.35
"""JP""",3433,3481,6914,50.35
"""EP""",5355,201,5556,3.62
"""DE""",1266,72,1338,5.38
"""CA""",1210,95,1305,7.28
"""AU""",1203,95,1298,7.32


## 3. Boundary Subclasses Audit

Per Note 2 of the `B64U` scheme and the `G08G 5/57` reclassification warning (which notes pending document transfers from `G08G 5/51`, `5/54`, and `5/56`), autonomous vehicle control (`G05D 1/00`) and air traffic management (`G08G 5/00`) contain UAV invetions that are not dual-tagged in `B64U`.

Because these classes also cover manned aviation and ground vehicles, we compute the conditional UAV purity (share of families in each subgroup co-occurring with our core UAV set).

In [3]:
clean_cpc = func.replace(TLS224_APPLN_CPC.cpc_class_symbol, " ", "")
q_boundary = (
    db.query(TLS201_APPLN.docdb_family_id, clean_cpc.label("cpc_symbol"))
        .select_from(TLS224_APPLN_CPC)
        .join(TLS201_APPLN, TLS224_APPLN_CPC.appln_id == TLS201_APPLN.appln_id)
        .filter(
            TLS201_APPLN.ipr_type == IPR_TYPE,
            TLS201_APPLN.earliest_filing_year.between(*YEAR_WINDOW),
            # boundary filters
            or_(clean_cpc.like("G05D1/%"), clean_cpc.like("G08G5/%")),
            ~clean_cpc.like("G08G5/57%"),
            ~clean_cpc.like("G08G5/27%"),
            ~clean_cpc.like("G05D1/228%"),
        )
        .distinct()
)

df_boundary = pl.from_pandas(patstat.df(q_boundary))

subgroup_audit = (
    df_boundary
        .with_columns(
            pl.col("cpc_symbol").str.slice(0, 8).alias("cpc_subgroup"),
            pl.col("docdb_family_id").is_in(core_family_ids["docdb_family_id"]).alias("is_core_uav"),
        )
        .group_by("cpc_subgroup")
        .agg(
            pl.col("docdb_family_id").n_unique().alias("total_families"),
            pl.col("docdb_family_id").filter(pl.col("is_core_uav")).n_unique().alias("core_uav_overlap"),
        )
        # sanity filter
        .filter(pl.col("total_families") >= 150)
        .with_columns(
            (pl.col("core_uav_overlap") / pl.col("total_families") * 100)
            .round(2)
            .alias("uav_purity_pct")
        )
        .sort("uav_purity_pct", descending=True)
)

subgroup_audit

cpc_subgroup,total_families,core_uav_overlap,uav_purity_pct
str,u32,u32,f64
"""G08G5/59""",1280,921,71.95
"""G08G5/55""",5803,3649,62.88
"""G08G5/32""",1645,1000,60.79
"""G08G5/22""",2506,1371,54.71
"""G08G5/58""",631,338,53.57
…,…,…,…
"""G05D1/24""",12138,911,7.51
"""G05D1/64""",8531,463,5.43
"""G05D1/02""",40362,927,2.3


### Decisions for Query Contruction

- **Union CPC and IPC:** Including IPC alongside CPC recovers over 13,000 substantive families missed due to reclassification lags across China ($22.8\%$), Japan ($50.4\%$), and South Korea ($18.4\%$), bringing the unconditional core to 61,967 families.

- **Purity-Thresholded Boundary Pool:** Promoting `G08G 5/57` and `G08G 5/27` to core reveals that adjacent air-traffic subgroups reach $53\%–72\%$ UAV purity (led by geofencing `G08G 5/59` at $72.0\%$ and single-aircraft guidance `G08G 5/55` at $62.9\%$).

---

## 4. Boundary Pool & Semantic Query Calibration

To retrieve un-reclassified UAV inventions inside the retained boundary subgroups (purity $\ge 20\%$), we construct a balanced benchmark pool of 2,000 families with English titles and abstracts (1,000 core-verified UAV families and 1,000 unlabeled boundary families).

Using a stratified evaluation split, we benchmark deterministic keywords against two semantic classifiers built on Patent SBERT embeddings (*graziasveva93/sbert-adapt-ub Ascione & Sterzi, 2024*):
1. **Constrative Centroid:** Cosine similarity to the UAV prototype minus 0.5* similarity to the unlabeled boundary prototype.
2. **Logistic Regression Probe:** A linear classifier that down-weights shared aviation control dimensions and isolates UAV-specific latent features, with decision thresholds calibrated via 5-fold out-of-fold predictions to maximize F0.5.

In [4]:
POOL_FILE = (CACHE_DIR / f"boundary_pool_p20_n2000.parquet")

valid_subgroups = subgroup_audit.filter(pl.col("uav_purity_pct") >= 20.0)["cpc_subgroup"]
valid_subgroups

cpc_subgroup
str
"""G08G5/59"""
"""G08G5/55"""
"""G08G5/32"""
"""G08G5/22"""
"""G08G5/58"""
…
"""G05D1/10"""
"""G08G5/21"""
"""G05D1/22"""


In [5]:
if POOL_FILE.exists():
    df_pool = pl.read_parquet(POOL_FILE)
else:
    boundary_fams = (
        df_boundary
            .with_columns(pl.col("cpc_symbol").str.slice(0, 8).alias("cpc_subgroup"))
            .filter(pl.col("cpc_subgroup").is_in(valid_subgroups))
            .select("docdb_family_id")
            .unique()
            .with_columns(pl.col("docdb_family_id").is_in(core_family_ids["docdb_family_id"]).alias("is_uav"))
    )
    
    # extracts 2,000 positive samples and 2,000 negative samples
    # note: we only need 1,000 each, but we have to filter them, so this is a "safety check"
    candidates = pl.concat(
        [
            boundary_fams.filter(pl.col("is_uav")).sample(n=2000, seed=42),
            boundary_fams.filter(~pl.col("is_uav")).sample(n=2000, seed=42)
        ]
    )
    
    q_text = (
        db.query(
            TLS201_APPLN.docdb_family_id,
            TLS202_APPLN_TITLE.appln_title.label("title"),
            TLS203_APPLN_ABSTR.appln_abstract.label("abstract")
        )
            .select_from(TLS201_APPLN)
            .join(TLS202_APPLN_TITLE, TLS202_APPLN_TITLE.appln_id == TLS201_APPLN.appln_id)
            .join(TLS203_APPLN_ABSTR, TLS203_APPLN_ABSTR.appln_id == TLS201_APPLN.appln_id)
            .filter(
                TLS201_APPLN.docdb_family_id.in_(candidates["docdb_family_id"].to_list()),
                TLS202_APPLN_TITLE.appln_title_lg == "en",
                TLS203_APPLN_ABSTR.appln_abstract_lg == "en"
            )
    )
    
    df_raw_text = (
        pl.from_pandas(patstat.df(q_text))
            .unique(subset=["docdb_family_id"], keep="first")
            .join(candidates, on="docdb_family_id", how="inner")
            .with_columns(pl.concat_str([pl.col("title"), pl.lit(". "), pl.col("abstract")]).alias("text"))
    )
    
    df_pool = pl.concat(
        [
            df_raw_text.filter(pl.col("is_uav")).sample(n=1000, seed=42),
            df_raw_text.filter(~pl.col("is_uav")).sample(n=1000, seed=42),
        ]
    )
    df_pool.write_parquet(POOL_FILE)

print(f"Boundary pool loaded: {len(df_pool):,} families across {len(valid_subgroups)} subgroups")
df_pool.group_by("is_uav").len()

Boundary pool loaded: 2,000 families across 32 subgroups


is_uav,len
bool,u32
true,1000
false,1000


In [6]:
POS_KEYWORDS = [
    r"\buav\b",
    r"\buavs\b",
    r"\buas\b",
    r"\brpas\b",
    r"\bunmanned aerial\b",
    r"\buncrewed aerial\b",
    r"\bremotely piloted\b",
    r"\bdrone\b",
    r"\bdrones\b",
    r"\bquadrotor\b",
    r"\bquadcopter\b",
    r"\bmultirotor\b",
    r"\bmulticopter\b",
    r"\bhexacopter\b",
    r"\boctocopter\b",
    r"\bevtol\b",
    r"\brotorcraft\b",
    r"\bflying robot\b",
]

NEG_KEYWORDS = [
    r"\bautonomous driving\b",
    r"\broad vehicle\b",
    r"\blawnmower\b",
    r"\bsubmersible\b",
    r"\bunderwater vehicle\b",
    r"\bauv\b",
    r"\busv\b",
]

Note: These are the sets of positive and negative keywords that we will be using. Some were generated using [Gemini Argon 4](https://aistudio.google.com/docs/models/gemini-4-argon), while others were selected following a review of relevant academic papers.

In [7]:
def match_keywords(
    texts: pl.Series, 
    pos_kw: list[str],
    neg_kw: list[str]
) -> np.ndarray:
    positive = "(?i)(" + "|".join(pos_kw) + ")"
    negative = "(?i)(" + "|".join(neg_kw) + ")"
    return (texts.str.contains(positive) & ~texts.str.contains(negative)).to_numpy()

# Contrastive semantic score
# cosine similarity to the UAV centroid minus 0.5* similarity to the non-UAV background centroid.
def compute_sbert_score(emb: np.ndarray) -> np.ndarray:
    return (emb @ uav_proto.T).ravel() - 0.5 * (emb @ bg_proto.T).ravel()

def compute_f05(pred: np.ndarray, target: np.ndarray) -> float:
    tp = (pred & target).sum()
    prec = tp / max(pred.sum(), 1)
    rec = tp / max(target.sum(), 1)
    return (1.25 * prec * rec) / max(0.25 * prec + rec, 1e-9)

def evaluate(name: str, p_eval: np.ndarray, t_eval: np.ndarray) -> dict:
    tp = int((p_eval & t_eval).sum())
    prec = tp / max(int(p_eval.sum()), 1)
    rec = tp / max(int(t_eval.sum()), 1)
    f05 = (1.25 * prec * rec) / max(0.25 * prec + rec, 1e-9)
    f1 = (2 * prec * rec) / max(prec + rec, 1e-9)
    return {
        "query": name,
        "selected": int(p_eval.sum()),
        "precision": round(prec, 3),
        "recall": round(rec, 3),
        "F0.5": round(f05, 3),
        "F1": round(f1, 3),
    }

In [8]:
# deterministic baseline
by_kw = match_keywords(df_pool["text"], POS_KEYWORDS, NEG_KEYWORDS)

# ground truth
truth = df_pool["is_uav"].to_numpy()

patent_sbert = SentenceTransformer("graziasveva93/sbert-adapt-ub")
embeddings = patent_sbert.encode(
    df_pool["text"].to_list(),
    batch_size=64,
    show_progress_bar=True,
    normalize_embeddings=True,
)

# shapes: emb (400, 384) / (1600, 384) | y & kw are True/False vectors (400,) / (1600,)
emb_seed, emb_eval, y_seed, y_eval, kw_seed, kw_eval = train_test_split(
    embeddings,
    truth,
    by_kw,
    # since sbert-adapt-ub is a pre-trained model 
    # we shouldnt need a large training set (20%)
    train_size=.2,
    stratify=truth,
    random_state=42
)

uav_proto = normalize(emb_seed[y_seed].mean(axis=0, keepdims=True))
bg_proto = normalize(emb_seed[~y_seed].mean(axis=0, keepdims=True))

# sbert_score measures how closely a patent aligns with the UAV
# semantic centroid relative to the non-UAV background centroid
seed_score = compute_sbert_score(emb_seed)
eval_score = compute_sbert_score(emb_eval)

# candidate tau (decision threshold)
grid = np.linspace(np.percentile(seed_score, 10), np.percentile(seed_score, 95), 200)
# the 'grid' value that get the highest f0.5-score
best_tau = max(
    grid, 
    key=lambda t: compute_f05(seed_score >= t, y_seed)
)
sbert_eval = eval_score >= best_tau

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Batches:   0%|          | 0/32 [00:00<?, ?it/s]

In [9]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_predict

# --- New Try: using a Logistic Regression model to fine-tune the tau ---
lr = LogisticRegression(C=0.2, max_iter=1000, random_state=42)

seed_prob_lr = cross_val_predict(
    lr, 
    emb_seed, y_seed, 
    cv=5, 
    method="predict_proba"
)[:, 1]
best_tau_lr = max(
    np.linspace(0.1, 0.9, 200), 
    key=lambda t: compute_f05(seed_prob_lr >= t, y_seed)
)

lr.fit(emb_seed, y_seed)
eval_prob_lr = lr.predict_proba(emb_eval)[:, 1]
sbert_lr_eval = eval_prob_lr >= best_tau_lr

In [10]:
pl.DataFrame([
    evaluate("Keywords", kw_eval, y_eval),
    evaluate("Patent-SBERT", sbert_eval, y_eval),
    evaluate("Keywords OR SBERT", kw_eval | sbert_eval, y_eval),
    evaluate("Keywords AND SBERT", kw_eval & sbert_eval, y_eval),
    evaluate("SBERT + LogReg", sbert_lr_eval, y_eval),
    evaluate("Keywords OR (SBERT+LogReg)", kw_eval | sbert_lr_eval, y_eval),
])

query,selected,precision,recall,F0.5,F1
str,i64,f64,f64,f64,f64
"""Keywords""",620,0.716,0.555,0.677,0.625
"""Patent-SBERT""",675,0.644,0.544,0.621,0.59
"""Keywords OR SBERT""",869,0.67,0.728,0.681,0.697
"""Keywords AND SBERT""",426,0.697,0.371,0.593,0.485
"""SBERT + LogReg""",734,0.718,0.659,0.705,0.687
"""Keywords OR (SBERT+LogReg)""",877,0.685,0.751,0.698,0.717


## 5. Semantic Term Discovery & Query Refinement

After running multiple tests across different pools, the results showed that the deterministic baseline (keywords) slightly outperforms all the alternatives.

We use `SBERT + LogReg` as a **semantic discovery tool** rather than a direct classifier. Specifically, we isolate true UAV patents captured by `SBERT + LogReg` but missed by `Keywords`, extract their most distinctive unigrams and bigrams via differential TF-IDF (filtering out patent boilerplate stopwords), and manually integrate high-precision synonyms (if needed) into `POS_KEYWORDS` before final validation on an unseen pool.

In [11]:
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS, TfidfVectorizer

PATENT_STOP_WORDS = {
    "said", "wherein", "whereby", "thereof", "therein", "comprising", "comprises",
    "comprise", "including", "includes", "plurality", "least", "configured",
    "provided", "providing", "present", "invention", "disclosed", "disclosure",
    "embodiment", "embodiments", "method", "methods", "device", "devices",
    "apparatus", "system", "systems", "unit", "first", "second", "third", "based",
}

_, df_eval = train_test_split(
    df_pool, 
    train_size=0.2, 
    stratify=truth, 
    random_state=42
)

# true UAV, only by SBERT + LogReg
only_semantic = sbert_lr_eval & ~kw_eval & y_eval
# true not-UAV
true_neg = ~sbert_lr_eval & ~kw_eval & ~y_eval

vec = TfidfVectorizer(
    stop_words=list(ENGLISH_STOP_WORDS | PATENT_STOP_WORDS),
    ngram_range=(1, 2),
    min_df=5,
    sublinear_tf=True,
)
X_eval_tfidf = vec.fit_transform(df_eval["text"].to_list())

diff_tfidf = np.asarray(
    X_eval_tfidf[only_semantic].mean(axis=0) - X_eval_tfidf[true_neg].mean(axis=0)
).ravel()
top_idx = diff_tfidf.argsort()[::-1][:15]

discovered_terms = pl.DataFrame({
    "discovered_ngram": vec.get_feature_names_out()[top_idx],
    "diff_tfidf_lift": diff_tfidf[top_idx].round(4),
})

print(f"True UAVs caught by SBERT+LogReg but missed by Keywords: {len(df_eval.filter(pl.Series(only_semantic)))}")
with pl.Config(tbl_rows=15):
    display(discovered_terms)

True UAVs caught by SBERT+LogReg but missed by Keywords: 157


discovered_ngram,diff_tfidf_lift
str,f64
"""flying""",0.0384
"""flight""",0.0327
"""aircraft""",0.0247
"""unmanned""",0.0227
"""unmanned aircraft""",0.0221
"""aerial""",0.017
"""flying body""",0.0156
"""aerial vehicle""",0.013
"""imaging""",0.0127


In [12]:
REFINED_POS_KEYWORDS = [
    *POS_KEYWORDS,
    r"\bunmanned aircraft\b",
    r"\bunmanned air\b",
    r"\bunmanned flight\b",
    r"\bunmanned flying\b",
    r"\bflying body\b",
    r"\bflying bodies\b",
    r"\bflying vehicle\b",
    r"\bflight vehicle\b",
    r"\baerial vehicle\b",
    r"\baerial vehicles\b",
    r"\baerial platform\b",
    r"\bvtol\b",
    r"\bmav\b",
    r"\bmicro air\b",
]

kw_refined_eval = match_keywords(df_eval["text"], REFINED_POS_KEYWORDS, NEG_KEYWORDS)

pl.DataFrame([
    evaluate("Keywords (Baseline)", kw_eval, y_eval),
    evaluate("Keywords (Refined)", kw_refined_eval, y_eval),
    evaluate("SBERT + LogReg", sbert_lr_eval, y_eval),
])

query,selected,precision,recall,F0.5,F1
str,i64,f64,f64,f64,f64
"""Keywords (Baseline)""",620,0.716,0.555,0.677,0.625
"""Keywords (Refined)""",712,0.746,0.664,0.728,0.702
"""SBERT + LogReg""",734,0.718,0.659,0.705,0.687


## 6. Family Validation Phase ($n = 100$)

Our final query unions two disjoint sets:
1. **Set 1 (Core Classification):** Families in the unconditional core CPC/IPC set.
2. **Set 2 (Boundary Rescue):** Non-core families in the high-purity boundary subgroups whose English title or abstract matches `REFINED_POS_KEYWORDS` (and excludes `NEG_KEYWORDS`).

We draw a stratified random sample of $n = 100$ families for manual inspection and compute the Stratified Cochran precision estimator with 95% Wilson confidence intervals.

In [13]:
VALIDATION_FILE = Path("../data/sample/step0_validation_100.csv")
VALIDATION_FILE.parent.mkdir(parents=True, exist_ok=True)

non_core_family_ids = (
    df_boundary
        .with_columns(pl.col("cpc_symbol").str.slice(0, 8).alias("cpc_subgroup"))
        .filter(pl.col("cpc_subgroup").is_in(valid_subgroups))
        .select("docdb_family_id")
        .unique()
        .filter(~pl.col("docdb_family_id").is_in(core_family_ids["docdb_family_id"]))
)

clean_cpc = func.replace(TLS224_APPLN_CPC.cpc_class_symbol, " ", "")
q_rescue = (
    db.query(
        TLS201_APPLN.docdb_family_id,
        TLS202_APPLN_TITLE.appln_title.label("title"),
        TLS203_APPLN_ABSTR.appln_abstract.label("abstract")
    )
        .select_from(TLS201_APPLN)
        .join(TLS202_APPLN_TITLE, TLS202_APPLN_TITLE.appln_id == TLS201_APPLN.appln_id)
        .join(TLS203_APPLN_ABSTR, TLS203_APPLN_ABSTR.appln_id == TLS201_APPLN.appln_id)
        .join(TLS224_APPLN_CPC, TLS224_APPLN_CPC.appln_id == TLS201_APPLN.appln_id)
        .filter(
            TLS201_APPLN.ipr_type == IPR_TYPE, # == "PI"
            TLS201_APPLN.earliest_filing_year.between(*YEAR_WINDOW), # 2004-2025
            TLS202_APPLN_TITLE.appln_title_lg == "en",
            TLS203_APPLN_ABSTR.appln_abstract_lg == "en",
            or_(*(clean_cpc.like(f"{g}%") for g in valid_subgroups.to_list())),
        )
)

df_boundary_texts = (
    pl.from_pandas(patstat.df(q_rescue))
        .unique(subset=["docdb_family_id"], keep="first")
        .join(non_core_family_ids, on="docdb_family_id", how="inner")
        .with_columns(pl.concat_str([pl.col("title"), pl.col("abstract")], separator=". ").alias("text"))
)

rescue_mask = match_keywords(df_boundary_texts["text"], REFINED_POS_KEYWORDS, NEG_KEYWORDS)
df_rescue = df_boundary_texts.filter(pl.Series(rescue_mask))

n1 = len(core_family_ids)
n2 = len(df_rescue)
print(f"Set 1 (Core): {n1:,} | Set 2 (Boundary Rescue): {n2:,} | Total: {n1 + n2:,}")

Set 1 (Core): 61,967 | Set 2 (Boundary Rescue): 9,220 | Total: 71,187


In [14]:
if not VALIDATION_FILE.exists():
    core_candidates = core_family_ids.sample(n=150, seed=42)["docdb_family_id"].to_list()
    
    q_core_text = (
        db.query(
            TLS201_APPLN.docdb_family_id,
            TLS202_APPLN_TITLE.appln_title.label("title"),
            TLS203_APPLN_ABSTR.appln_abstract.label("abstract"),
        )
            .select_from(TLS201_APPLN)
            .join(TLS202_APPLN_TITLE, TLS202_APPLN_TITLE.appln_id == TLS201_APPLN.appln_id)
            .join(TLS203_APPLN_ABSTR, TLS203_APPLN_ABSTR.appln_id == TLS201_APPLN.appln_id)
            .filter(
                TLS201_APPLN.docdb_family_id.in_(core_candidates),
                TLS202_APPLN_TITLE.appln_title_lg == "en",
                TLS203_APPLN_ABSTR.appln_abstract_lg == "en",
            )
    )
    
    set1 = (
        pl.from_pandas(patstat.df(q_core_text))
            .unique(subset=["docdb_family_id"], keep="first")
            .sample(n=50, seed=42)
            .with_columns(pl.lit("core").alias("set"), pl.lit(1).alias("is_relevant"))
    )
    set2 = (
        df_rescue.select("docdb_family_id", "title", "abstract")
            .sample(n=50, seed=42)
            .with_columns(pl.lit("boundary_rescue").alias("set"), pl.lit(1).alias("is_relevant"))
    )
    
    pl.concat([set1, set2]).write_csv(VALIDATION_FILE)
print(f"Saved 100-family stratified sample to {VALIDATION_FILE}")

Saved 100-family stratified sample to ../data/sample/step0_validation_100.csv


In [15]:
from scipy.stats import norm

def wilson_ci(k: int, n: int, alpha: float = 0.05) -> dict:
    # sample precision
    p_hat = k / n
    # z-score (bidirectional)
    z = norm.ppf(1 - alpha / 2)
    
    denom = 1 + z**2 / n
    center = (p_hat + z**2 / (2 * n)) / denom
    half = z * np.sqrt((p_hat * (1 - p_hat) + z**2 / (4 * n)) / n) / denom
    return {
        "n": n,
        "k_true": k,
        "precision": round(p_hat, 4),
        "wilson_95_low": round(max(0.0, center - half), 4),
        "wilson_95_high": round(min(1.0, center + half), 4)
    }


df_val = pl.read_csv("../data/sample/step0_validated_100.csv")

# Stratified Cochran
strata_pop = {"core": n1, "boundary_rescue": n2}

rows, p_strat, var_strat = [], 0.0, 0.0
for name, N_h in strata_pop.items():
    sub = df_val.filter(pl.col("set") == name)
    n_h, k_h = len(sub), int(sub["is_relevant"].sum())
    w_h, p_h = N_h / (n1+n2), k_h / n_h
    p_strat += w_h * p_h
    var_strat += (w_h**2) * (1 - n_h / N_h) * (p_h * (1 - p_h)) / max(n_h - 1, 1)
    rows.append({
        "set": name,
        "pop_weight": round(w_h, 4),
        **wilson_ci(k_h, n_h),
    })

se_strat = np.sqrt(var_strat)
z95 = norm.ppf(0.975)
rows.append({
    "set": "Stratified Cochran",
    "pop_weight": 1.0,
    "n": len(df_val),
    "k_true": int(df_val["is_relevant"].sum()),
    "precision": round(p_strat, 4),
    "wilson_95_low": round(max(0.0, p_strat - z95 * se_strat), 4),
    "wilson_95_high": round(min(1.0, p_strat + z95 * se_strat), 4)
})

pl.DataFrame(rows)

set,pop_weight,n,k_true,precision,wilson_95_low,wilson_95_high
str,f64,i64,i64,f64,f64,f64
"""core""",0.8705,50,43,0.86,0.7381,0.9305
"""boundary_rescue""",0.1295,50,49,0.98,0.895,0.9965
"""Stratified Cochran""",1.0,100,92,0.8755,0.7909,0.9602
